# Regresión Lineal, Lasso, Ridge y ElasticNet

Template para modelado predictivo basado en regresión. Incluye los pasos de desarrollo del modelo para predicción de precios de acciones.

## Contenido

* [1. Planteamiento del Problema](#0)
* [2. Carga de Librerías y Dataset](#1)
    * [2.1. Cargar Librerías](#1.1)
    * [2.2. Cargar Dataset](#1.2)
* [3. Análisis Exploratorio de Datos](#2)
* [4. Preparación de Datos](#3)
    * [4.1. División Entrenamiento/Validación](#3.1)
* [5. Evaluación de Modelos](#4)
    * [5.1. Comparación de Modelos](#4.1)
    * [5.2. Grid Search](#4.2)
* [6. Finalización del Modelo](#5)

<a id='0'></a>
# 1. Planteamiento del Problema

El objetivo de este notebook es comprender cómo trabajar un problema de modelado predictivo de principio a fin. Se presenta un caso de estudio de predicción de precios de acciones.

**Modelos implementados:**

* Regresión Lineal
* Lasso (regularización L1)
* Ridge (regularización L2)
* ElasticNet (combinación L1 + L2)

<a id='1'></a>
# 2. Carga de Librerías y Dataset

<a id='1.1'></a>
## 2.1. Cargar Librerías

In [ ]:
# Cargar librerías
import numpy as np
import pandas as pd
import pandas_datareader.data as web
from matplotlib import pyplot

from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge
from sklearn.linear_model import ElasticNet

from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

import warnings
warnings.filterwarnings('ignore')

<a id='1.2'></a>
## 2.2. Cargar Dataset

In [ ]:
# Obtener datos usando pandas_datareader
return_period = 21

stk_tickers = ['MSFT', 'IBM', 'GOOGL']
ccy_tickers = ['DEXJPUS', 'DEXUSUK']
idx_tickers = ['SP500', 'DJIA', 'VIXCLS']

stk_data = web.DataReader(stk_tickers, 'yahoo')
ccy_data = web.DataReader(ccy_tickers, 'fred')
idx_data = web.DataReader(idx_tickers, 'fred')

Y = np.log(stk_data.loc[:, ('Adj Close', 'MSFT')]).diff(return_period).shift(-return_period)
Y.name = Y.name[-1]+'_pred'

X1 = np.log(stk_data.loc[:, ('Adj Close', ('GOOGL', 'IBM'))]).diff(return_period)
X1.columns = X1.columns.droplevel()
X2 = np.log(ccy_data).diff(return_period)
X3 = np.log(idx_data).diff(return_period)

X4 = pd.concat([Y.diff(i) for i in [21, 63, 126, 252]], axis=1).dropna()
X4.columns = ['1M', '3M', '6M', '1Y']

X = pd.concat([X1, X2, X3, X4], axis=1)

dataset = pd.concat([Y, X], axis=1).dropna()
Y = dataset.loc[:, Y.name]
X = dataset.loc[:, X.columns]

### Conversión a formato de regresión supervisada

Las variables predictoras se transforman en variables rezagadas (lagged). El valor t-1 de cada variable se utiliza para la predicción.

In [ ]:
def series_to_supervised(data, lag=1):
    n_vars = data.shape[1]
    df = pd.DataFrame(data)
    cols, names = list(), list()
    for i in range(lag, 0, -1):
        cols.append(df.shift(i))
        names += [('%s(t-%d)' % (df.columns[j], i)) for j in range(n_vars)]
    agg = pd.concat(cols, axis=1)
    agg.columns = names
    agg = pd.DataFrame(data.iloc[:, 0]).join(agg)
    agg.dropna(inplace=True)
    return agg

dataset = series_to_supervised(dataset, 1)

In [ ]:
# Vista previa de los datos
dataset.head()

<a id='2'></a>
# 3. Análisis Exploratorio de Datos

In [ ]:
# Forma del dataset
print("Dimensiones:", dataset.shape)

# Estadísticas descriptivas
dataset.describe()

<a id='3'></a>
# 4. Preparación de Datos

In [ ]:
# Separar variables objetivo (Y) y predictoras (X)
Y = dataset.iloc[:, 0]
X = dataset.iloc[:, 1:]

<a id='3.1'></a>
## 4.1. División Entrenamiento/Validación

In [ ]:
# División secuencial para series de tiempo
validation_size = 0.2
seed = 7

train_size = int(len(X) * (1 - validation_size))
X_train, X_validation = X[0:train_size], X[train_size:len(X)]
Y_train, Y_validation = Y[0:train_size], Y[train_size:len(X)]

print(f"Tamaño entrenamiento: {len(X_train)}")
print(f"Tamaño validación: {len(X_validation)}")

<a id='4'></a>
# 5. Evaluación de Modelos

## 5.1. Opciones de evaluación

In [ ]:
# Opciones para regresión
num_folds = 10
scoring = 'neg_mean_squared_error'

<a id='4.1'></a>
## 5.1. Comparación de Modelos

In [ ]:
# Modelos de regresión lineal con regularización
models = []
models.append(('LR', LinearRegression()))
models.append(('LASSO', Lasso()))
models.append(('RIDGE', Ridge()))
models.append(('EN', ElasticNet()))

In [ ]:
# Validación cruzada K-fold
results = []
names = []
for name, model in models:
    kfold = KFold(n_splits=num_folds, random_state=seed)
    cv_results = -1 * cross_val_score(model, X_train, Y_train, cv=kfold, scoring=scoring)
    results.append(cv_results)
    names.append(name)
    msg = "%s: %f (%f)" % (name, cv_results.mean(), cv_results.std())
    print(msg)

In [ ]:
# Comparación visual de algoritmos
fig = pyplot.figure()
fig.suptitle('Comparación de Algoritmos (MSE)')
ax = fig.add_subplot(111)
pyplot.boxplot(results)
ax.set_xticklabels(names)
fig.set_size_inches(10, 6)
pyplot.ylabel('Error Cuadrático Medio')
pyplot.show()

El gráfico muestra el MSE. Cuanto menor sea el MSE, mejor es el rendimiento del modelo.

<a id='4.2'></a>
## 5.2. Grid Search

In [ ]:
# 1. Grid Search: Regresión Lineal
param_grid = {'fit_intercept': [True, False]}
model = LinearRegression()
kfold = KFold(n_splits=num_folds, random_state=seed)
grid = GridSearchCV(estimator=model, param_grid=param_grid, scoring=scoring, cv=kfold)
grid_result = grid.fit(X_train, Y_train)
print("Mejor: %f usando %s" % (grid_result.best_score_, grid_result.best_params_))
for mean, stdev, param in zip(grid_result.cv_results_['mean_test_score'],
                               grid_result.cv_results_['std_test_score'],
                               grid_result.cv_results_['params']):
    print("%f (%f) con: %r" % (mean, stdev, param))

In [ ]:
# 2. Grid Search: Lasso
# alpha: constante que multiplica el término L1. alpha=0 es equivalente a OLS.
param_grid = {'alpha': [0.01, 0.1, 0.3, 0.7, 1, 1.5, 3, 5]}
model = Lasso()
kfold = KFold(n_splits=num_folds, random_state=seed)
grid = GridSearchCV(estimator=model, param_grid=param_grid, scoring=scoring, cv=kfold)
grid_result = grid.fit(X_train, Y_train)
print("Mejor: %f usando %s" % (grid_result.best_score_, grid_result.best_params_))
for mean, stdev, param in zip(grid_result.cv_results_['mean_test_score'],
                               grid_result.cv_results_['std_test_score'],
                               grid_result.cv_results_['params']):
    print("%f (%f) con: %r" % (mean, stdev, param))

In [ ]:
# 3. Grid Search: Ridge
# alpha: constante que multiplica el término L2.
param_grid = {'alpha': [0.01, 0.1, 0.3, 0.7, 1, 1.5, 3, 5]}
model = Ridge()
kfold = KFold(n_splits=num_folds, random_state=seed)
grid = GridSearchCV(estimator=model, param_grid=param_grid, scoring=scoring, cv=kfold)
grid_result = grid.fit(X_train, Y_train)
print("Mejor: %f usando %s" % (grid_result.best_score_, grid_result.best_params_))
for mean, stdev, param in zip(grid_result.cv_results_['mean_test_score'],
                               grid_result.cv_results_['std_test_score'],
                               grid_result.cv_results_['params']):
    print("%f (%f) con: %r" % (mean, stdev, param))

In [ ]:
# 4. Grid Search: ElasticNet
# alpha: constante que multiplica los términos de penalización
# l1_ratio: parámetro de mezcla (0=L2, 1=L1, entre 0-1=combinación)
param_grid = {'alpha': [0.01, 0.1, 0.3, 0.7, 1, 1.5, 3, 5],
              'l1_ratio': [0.01, 0.1, 0.3, 0.5, 0.7, 0.9, 0.99]}
model = ElasticNet()
kfold = KFold(n_splits=num_folds, random_state=seed)
grid = GridSearchCV(estimator=model, param_grid=param_grid, scoring=scoring, cv=kfold)
grid_result = grid.fit(X_train, Y_train)
print("Mejor: %f usando %s" % (grid_result.best_score_, grid_result.best_params_))
for mean, stdev, param in zip(grid_result.cv_results_['mean_test_score'],
                               grid_result.cv_results_['std_test_score'],
                               grid_result.cv_results_['params']):
    print("%f (%f) con: %r" % (mean, stdev, param))

<a id='5'></a>
# 6. Finalización del Modelo

In [ ]:
# Entrenar el modelo con los mejores parámetros (ejemplo: Regresión Lineal)
model = LinearRegression(fit_intercept=True)
model.fit(X_train, Y_train)

In [ ]:
# Evaluar en el conjunto de validación
predictions = model.predict(X_validation)
print("Error Cuadrático Medio (MSE):", mean_squared_error(Y_validation, predictions))
print("R² (coeficiente de determinación):", r2_score(Y_validation, predictions))

In [ ]:
# Comparar predicciones vs valores reales
comparison = pd.DataFrame({'Real': Y_validation.values, 'Predicho': predictions})
comparison.head(10)